# 21: Parsing Files

## Reading and writing from files

The fist part of this notebook works with a FASTA file of the SARS-Cov-2 reference genome. The source is here: https://www.ncbi.nlm.nih.gov/nuccore/NC_045512  You can download the file `covid-ref.fsa` from the course repository at: 
https://github.com/Bio724D/Bio724D_2025_2026/tree/main/data

The `open()` function is the standard way to gain access to a file on your filesystem. It returns a file object, which can be assigned to an identifier.   
   
The primary argument to the `open()` function is a string giving the path and name of the file you want to open. Files can be opened for reading only, writing, or both reading and writing. The default setting is to open the file in read-only mode, or `'r'`. The code below makes this explicit. Other modes are: write-only `'w'`, append `'a'`, and read and write `'r+'`.

In [112]:
# open the file in read-only mode and assign


Now that we have a file object, we can use methods like `read()` and `write()` to interact with it. You can read more about `open()` [**here**](https://docs.python.org/3/library/functions.html#open) and about file objects [**here**](https://docs.python.org/3/glossary.html#term-file-object). 

In [113]:
# what class is a file object?


_io.TextIOWrapper

The simplest way to manipulate a file is simply to read all the information from it and return the data in the file as a string. The `read()` function performs this for you. Note that it reads *all* the data into your computer's memory. If the input was a very large file this could be a problem.

In [4]:
# reading everything from a file as a single string


In [1]:
# what kind of object is s?


In [2]:
# take a look a the first 300 characters


In [3]:
# how long is the entire string


Once you've read what you need from the file it's good practice to close it (failing to close a file can lead to a memory leak in some contexts, but it's usually not a problem in an interactive environment like Jupyter notebooks).

In [118]:
f.close()

A safer way to read a file is within a `with` statement as illustrated below.  The advantage of the `with` statement is it insures the file is closed (i.e. you don't need to explicitly call the `close()` method) at the end of the `with` block. This is an example of a context manager, an object that guarantees that specific resources are setup *and* closed down, even if errors occur during a program's execution.

Once we've read the file into a string we can apply all the standard string methods and operators to it:

In [5]:
# how many characters? (compare to above)


In [7]:
# view the first 50 chars


In [6]:
# and the last 50 characters


## Reading a file by lines

Sometimes it's more convenient or more efficient to get the information in the file in terms of lines.  The `readlines()` method associated with file object read's all the lines at once in a list:

In [8]:
## return a list of the lines in the file


In [ ]:
# what kind of data object?

In [ ]:
# how many lines?


The first line of an entry in a FASTA file is a "header", followed by 1 or more lines of sequence. Header lines start with a `>` (right bracket) symbol. The SARS-CoV-2 genome FASTA only contains one entry. 

In [9]:
# show the "header" line for the reference genome


In [10]:
# subsequent lines are the actual sequence data


Notice the newline character (`\n`) at the end of each line.

## Reading a file one line at a time

The `readlines()` function illustrated above reads all the lines at once. That's works well if your file has a modest number of lines, but for a file with millions of lines or with very long  lines,  `readlines()` might exhaust the memory of your computer.  One way to work around this is to process files line by line, reading only one line at a time. The code below uses a for loop to read each line and extract its length. Only one line at a time is loaded into memory.

In [31]:
# count the number of characters on each line



In [11]:
# view the first 10 counts


## Iterating through a file, filtering and concatenating lines

Here we illustrate the process of iterating through the lines of a file, doing some simple filtering, and concatenating lines into a single string. First, we tally the number of each base within the genome. Second, we concatenate the sequence, which in the `.fasta` file contains numerous newline characters.   

Some methods used in these code blocks:   
`items()` returns a key, value pair    
`strip()` removes newlines and other whitespace at beginning/end of lines    
`count()` returns a tally of non-overlapping occurrences of a given substring within the string   


In [33]:
# tally the number of bases in the SARS-Cov2 genome

# create an empty dictionary to hold the base tallies

# read lines one at a time


In [32]:
# view the tallies


In [21]:
# generate a string consisting of the SARS-Cov2 genome

# create an empty string to hold the concatenated sequence

# read lines one at a time


In [34]:
# total length of COVID reference genome nucleotide sequence
# does not include the header line which we filtered out in our for loop above


In [35]:
# first 100 characters in this sequence


## Extracting codons  using string slicing

Since the string we're working with represents the Sars-Cov-2 genome, let's extract a subsequence of interest that represents the gene that encodes the Spike protein. Once we've done so we'll generate  codons from that subsequence and translate them to their corresponding amino acids.

The spike protein coordinates from NCBI are given as: 21563..25384 but these are 1-indexed and inclusive of start/end coordinates. To extract the corresponding sequence from our Python string we need to convert these to 0-indexed coordinates and remember that Python indexing is up to but not including end index

In [36]:
# extract the DNA sequence of the gene encoding the spike protein


In [37]:
# length of sequence we extracted


In [38]:
# first 10 nucleotides


In [39]:
# last 10 nucleotides


In [40]:
# length divisible by 3? (should be True for coding sequence)


Next we'll extract the nucleotide triplets that represent the codons of the spike protein. This task is simple in this case because there are no introns to consider and the gene is encoded in the same strand orientation as the data is provided to us (not always the case). If the gene were on the opposite strand, we could use the function we wrote last week to find the reverse complement.

In [41]:
# create a list of the start positions for each codon

# convert to list() to see what it looks like, because range() returns an iterator


In [42]:
# extract codon subsequences by indexing with codon starts and slicing three characters

# take a look to make sure it worked as expected


Now that we have codons, the next step is to translate them. To do this, we will use the genetic code from the standard codon table given by NCBI (https://www.ncbi.nlm.nih.gov/Taxonomy/Utils/wprintgc.cgi#SG1). We will arrange this as a dictionary that maps from codons (the keys) to single-letter representations of amino acids (the values).

In [25]:
# Below is the standard code, cut and pasted from the NCBI website

Base1  = "TTTTTTTTTTTTTTTTCCCCCCCCCCCCCCCCAAAAAAAAAAAAAAAAGGGGGGGGGGGGGGGG"
Base2  = "TTTTCCCCAAAAGGGGTTTTCCCCAAAAGGGGTTTTCCCCAAAAGGGGTTTTCCCCAAAAGGGG"
Base3  = "TCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAG"
AAs    = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"

For example "TTT" (first column) translates to "F" (phenylalanine), "GGG" (last column) translates to "G" (glycine), "TAG" to "*" (stop codon), etc.   

Next, create a dictionary from those strings.

In [43]:
# turn strings into a dictionary

# create an empty dictionary

# loop over every column


In [44]:
## test our dictionary


Having set up this dictionary, we can "translate" the codons into a protein using a simple lookup process. 

In [45]:
# translate the spike protein

# create an empty list to hold the AA sequence

# loop over the codons, retrieving the AA


In [46]:
# show the first 10 AAs


Now, we just need to concatenate the individual strings into a single string. The `join()` method can do this: is combines strings in order, optionally with a separator. In the code below, we use `""` to indicate no separator.

In [47]:
# concatenate into a single string

# what happens if you write "!".join(spike_AAs) instead?

In [48]:
# look at the first 50 AAs


In [49]:
# and the last 50 AAs


## Parsing a FASTA file

The FASTA file format is the most commonly used file format used to represent nucleotide and protein sequence data.  Wikipedia has a good [overview of the FASTA format](https://en.wikipedia.org/wiki/FASTA_format).  

Summary of FASTA format:
 
 * Each file can hold one or more sequence records
 
 * The beginning of each record is delimited by a line called a header, which has a `>` character at the beginning, followed by the name associated with that record (and an optional description). For example `>seq1 Involved in...` would indicate the beginning of a record with the name `seq1` and the description "Involved in...".
 
 * One or more sequence lines follow each header line. Sequence lines are usually wrapped to have length <=80 characters but this is not required. 
     

Below is a simple function that will parse a FASTA file, returning each record as an element in a Python dictionary with the first word in the header line of each record as the key. 

This implementation isn't particularly robust or optimal, but illustrates some key aspects of flow-control in Python and parsing non-tabular data.


In [51]:
def parse_FASTA(fname):

    # open the file in read-only mode

    # initialize variables

    # read one line at a time from the file

        # for each line, there are three possibilities: empty, new record, or sequence
        
    # done reading lines

    # remember to close the file!

    # return the completed dictionary


To test our `parse_FASTA` function download the [`Spike-protein-aligned.fasta`](https://github.com/Bio724D/Bio724D_2023_2024/tree/main/data/Spike-protein-aligned.fasta) file  to your computer and modify the paths below as necessary to load and parse the sequence records contained in that file.

In [52]:
# load the data


In [53]:
# examine the type of object we got back


In [54]:
# how many records are there


In [55]:
# what are the keys of the dictionary of recrods


In [56]:
# get a specific record, and the first 80 AAs


In [57]:
# print the first 25 positions in the alignment for all the records


## Parsing .csv files

The code above illustrates how to parse a simple file format "from scratch". However, Python is a large and well developed ecosystem; many file formats you are likely to encounter (including FASTA) already have robust and thoroughly tested parsing libraries. In this last section, we'll parse `.csv` files.

In [58]:
# take a look at the file we will working with


This is hard to read and process!   

The code below illustrates how to parse comma-delimited or tab-delimited files using the built-in [`csv` module](https://docs.python.org/3/library/csv.html) that is part of Python's standard library.

The basic parsing tool is the `csv.reader` function which will parse each line in a file:

In [59]:
# read a .csv file into a list of lines


When using `csv.reader` the rows of the CSV file are returned as lists of strings.

An alternative "reader" is [`csv.DictReader`](https://docs.python.org/3/library/csv.html#csv.DictReader) which will return a list containing each row as a separate dictionary, with the keys being specified as the first line of the input file (the typical header line).  If your input file has no header line, you can specify a header with the `fieldnames` argument. 

In [60]:
# read a .csv file into a dictionary


Notice that `csv.DictReader()` returns each row as a dictionary, with the keys being the fieldnames.   
   
You can now index individual values by their key (like indexing using a column in a dataframe).

In [61]:
# print specific rows and columns
